<a href="https://colab.research.google.com/github/Sarthakpal23/fitness-tracker-/blob/main/fitness_analysis_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [164]:
!pip install streamlit
!pip install pyngrok


In [165]:
!pip install streamlit

In [166]:
%%writefile app.py
# ============================================================
# ULTIMATE FITNESS AI DASHBOARD
# Developed by: Sarthak Pal
# Instagram Style UI + ML + AI Diet Planner
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# PAGE CONFIG (ONLY ONCE)
# ------------------------------------------------------------
st.set_page_config(page_title="Fitness AI", page_icon="🔥", layout="wide")

# ------------------------------------------------------------
# INSTAGRAM STYLE UI
# ------------------------------------------------------------
st.markdown("""
<style>
html, body, [class*="css"] {
    background-color:#0f0f0f;
    color:white;
    font-family: 'Segoe UI', sans-serif;
}

.main-title {
    font-size:48px;
    font-weight:800;
    background: linear-gradient(90deg,#ff512f,#dd2476);
    -webkit-background-clip:text;
    -webkit-text-fill-color:transparent;
}

.metric-box {
    background: linear-gradient(135deg,#1f1c2c,#928dab);
    padding:20px;
    border-radius:15px;
    text-align:center;
    font-size:20px;
}

section[data-testid="stSidebar"] {
    background: linear-gradient(180deg,#141E30,#243B55);
}

.stButton>button {
    background: linear-gradient(90deg,#ff512f,#dd2476);
    border:none;
    border-radius:20px;
    color:white;
}
</style>
""", unsafe_allow_html=True)

st.markdown('<p class="main-title">🔥 Fitness AI Dashboard</p>', unsafe_allow_html=True)

# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------
@st.cache_data
def load_data():
    np.random.seed(42)
    data = pd.DataFrame({
        "Steps": np.random.randint(2000,15000,500),
        "Calories": np.random.randint(1500,3500,500),
        "Heart_Rate": np.random.randint(60,140,500),
        "Sleep": np.random.uniform(4,9,500),
        "BMI": np.random.uniform(18,32,500)
    })

    conditions = [
        (data["Steps"] < 4000),
        (data["Steps"] < 8000),
        (data["Steps"] < 12000),
        (data["Steps"] >= 12000)
    ]

    labels = ["Underactive","Moderate","Healthy","Highly Active"]
    data["Fitness_Level"] = np.select(conditions,labels)

    return data

df = load_data()

# ------------------------------------------------------------
# SIDEBAR
# ------------------------------------------------------------
st.sidebar.title("Navigation")

menu = st.sidebar.radio(
    "Select Page",
    ["Dashboard","Analytics","Prediction AI","AI Diet Planner","Health Report"]
)

# ------------------------------------------------------------
# FITNESS SCORE
# ------------------------------------------------------------
def fitness_score(steps, sleep, bmi):
    score = 0
    if steps > 10000:
        score += 40
    elif steps > 7000:
        score += 30
    else:
        score += 15

    if sleep >= 7:
        score += 30
    else:
        score += 15

    if 18.5 <= bmi <= 24.9:
        score += 30
    else:
        score += 15

    return score

# ------------------------------------------------------------
# DASHBOARD
# ------------------------------------------------------------
if menu == "Dashboard":

    col1,col2,col3,col4 = st.columns(4)

    col1.markdown(f'<div class="metric-box">Records<br>{len(df)}</div>', unsafe_allow_html=True)
    col2.markdown(f'<div class="metric-box">Avg Steps<br>{int(df["Steps"].mean())}</div>', unsafe_allow_html=True)
    col3.markdown(f'<div class="metric-box">Calories<br>{int(df["Calories"].mean())}</div>', unsafe_allow_html=True)
    col4.markdown(f'<div class="metric-box">Sleep<br>{round(df["Sleep"].mean(),2)}</div>', unsafe_allow_html=True)

    st.markdown("### Activity Distribution")

    fig, ax = plt.subplots()
    df["Fitness_Level"].value_counts().plot(kind="bar", ax=ax)
    st.pyplot(fig)

# ------------------------------------------------------------
# ANALYTICS
# ------------------------------------------------------------
elif menu == "Analytics":

    st.subheader("Dataset")
    st.dataframe(df)

    st.subheader("Correlation")

    corr = df.corr(numeric_only=True)
    fig, ax = plt.subplots()
    cax = ax.matshow(corr)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    st.pyplot(fig)

# ------------------------------------------------------------
# PREDICTION AI
# ------------------------------------------------------------
elif menu == "Prediction AI":

    col1,col2 = st.columns(2)

    with col1:
        steps = st.slider("Steps",1000,20000,7000)
        calories = st.slider("Calories",1000,4000,2200)
        heart = st.slider("Heart Rate",50,160,80)

    with col2:
        sleep = st.slider("Sleep",3.0,10.0,7.0)
        bmi = st.slider("BMI",15.0,35.0,23.0)

    if st.button("Analyze Fitness"):

        X = df.drop("Fitness_Level", axis=1)
        y = df["Fitness_Level"]

        X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

        model = RandomForestClassifier()
        model.fit(X_train,y_train)

        user = pd.DataFrame({
            "Steps":[steps],
            "Calories":[calories],
            "Heart_Rate":[heart],
            "Sleep":[sleep],
            "BMI":[bmi]
        })

        pred = model.predict(user)

        score = fitness_score(steps,sleep,bmi)

        st.success(f"Fitness Level: {pred[0]}")
        st.info(f"Fitness Score: {score}/100")

# ------------------------------------------------------------
# AI DIET PLANNER (COMBINED)
# ------------------------------------------------------------
elif menu == "AI Diet Planner":

    st.subheader("🥗 AI Diet + Smart Meal Generator")

    col1,col2,col3 = st.columns(3)

    with col1:
        bmi = st.slider("BMI",15.0,35.0,23.0)
        steps = st.slider("Daily Steps",1000,20000,8000)

    with col2:
        workout = st.slider("Workout Minutes",0,120,30)
        goal = st.selectbox("Goal",["Fat Loss","Muscle Gain","Maintain"])

    with col3:
        diet_type = st.selectbox("Diet Type",["Balanced","High Protein","Vegetarian","Low Carb"])

    if st.button("Generate Diet Plan"):

        calories = 1800
        calories += int(steps/2000)*50
        calories += int(workout/30)*100

        if goal == "Fat Loss":
            calories -= 300
        elif goal == "Muscle Gain":
            calories += 300

        st.success(f"Daily Calories Target: {calories}")

        breakfast = ["Oats","Eggs","Fruit"]
        lunch = ["Brown Rice","Chicken / Paneer","Vegetables"]
        snack = ["Protein Shake","Nuts"]
        dinner = ["Soup","Salad","Tofu / Fish"]

        if diet_type == "Vegetarian":
            lunch = ["Paneer","Quinoa","Vegetables"]
            dinner = ["Tofu","Salad"]

        if diet_type == "High Protein":
            breakfast = ["Egg Omelette","Protein Shake"]
            snack = ["Greek Yogurt","Almonds"]

        if diet_type == "Low Carb":
            lunch = ["Grilled Chicken","Broccoli"]
            dinner = ["Fish","Vegetables"]

        col4,col5 = st.columns(2)

        with col4:
            st.markdown("### Breakfast")
            for f in breakfast:
                st.write("•",f)

            st.markdown("### Lunch")
            for f in lunch:
                st.write("•",f)

        with col5:
            st.markdown("### Snack")
            for f in snack:
                st.write("•",f)

            st.markdown("### Dinner")
            for f in dinner:
                st.write("•",f)

# ------------------------------------------------------------
# HEALTH REPORT
# ------------------------------------------------------------
elif menu == "Health Report":

    st.subheader("Health Insights")

    st.write("Average Steps:", int(df["Steps"].mean()))
    st.write("Average Calories:", int(df["Calories"].mean()))
    st.write("Average Heart Rate:", int(df["Heart_Rate"].mean()))
    st.write("Average Sleep:", round(df["Sleep"].mean(),2))

# Footer
st.markdown("---")
st.write("Developed by Sarthak Pal | AI Project")


Overwriting app.py


In [192]:
%%writefile app.py
# ============================================================
# ULTIMATE FITNESS AI DASHBOARD
# Advanced UI + Machine Learning + Analytics
# Developed by: Sarthak Pal
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# Page Setup
# ------------------------------------------------------------
st.set_page_config(
    page_title="Fitness AI Dashboard",
    page_icon="🏃",
    layout="wide"
)

# ------------------------------------------------------------
# MODERN INSTAGRAM-STYLE UI
# ------------------------------------------------------------
st.set_page_config(
    page_title="Fitness AI",
    page_icon="🔥",
    layout="wide"
)

st.markdown("""
<style>

html, body, [class*="css"]  {
    font-family: 'Segoe UI', sans-serif;
    background-color: #0f0f0f;
    color: white;
}

/* Main Title */
.main-title {
    font-size:48px;
    font-weight:800;
    background: linear-gradient(90deg,#ff512f,#dd2476);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

/* Cards */
.card {
    background: rgba(255,255,255,0.05);
    padding: 25px;
    border-radius: 20px;
    backdrop-filter: blur(12px);
    border: 1px solid rgba(255,255,255,0.1);
    transition: 0.3s;
}

.card:hover {
    transform: translateY(-5px);
    box-shadow: 0px 10px 25px rgba(255,0,150,0.2);
}

/* Metrics */
.metric-box {
    background: linear-gradient(135deg,#1f1c2c,#928dab);
    padding: 20px;
    border-radius: 18px;
    text-align:center;
    font-size:20px;
    font-weight:600;
}

/* Sidebar */
section[data-testid="stSidebar"] {
    background: linear-gradient(180deg,#141E30,#243B55);
}

/* Buttons */
.stButton>button {
    background: linear-gradient(90deg,#ff512f,#dd2476);
    border: none;
    border-radius: 20px;
    padding: 10px 25px;
    color: white;
    font-weight: 600;
}

</style>
""", unsafe_allow_html=True)

st.markdown('<p class="main-title">🔥 Fitness AI Social Dashboard</p>', unsafe_allow_html=True)
st.write("AI Powered Health Insights • Smart Analytics • Personalized Fitness")


# ------------------------------------------------------------
# Dataset Generator
# ------------------------------------------------------------
@st.cache_data
def load_data():
    np.random.seed(42)

    data = pd.DataFrame({
        "Steps": np.random.randint(2000, 15000, 500),
        "Calories": np.random.randint(1500, 3500, 500),
        "Heart_Rate": np.random.randint(60, 140, 500),
        "Sleep": np.random.uniform(4, 9, 500),
        "BMI": np.random.uniform(18, 32, 500)
    })

    conditions = [
        (data["Steps"] < 4000),
        (data["Steps"] < 8000),
        (data["Steps"] < 12000),
        (data["Steps"] >= 12000)
    ]

    labels = ["Underactive", "Moderate", "Healthy", "Highly Active"]

    data["Fitness_Level"] = np.select(conditions, labels, default='Unknown')

    return data

df = load_data()

# ------------------------------------------------------------
# Sidebar Navigation
# ------------------------------------------------------------
st.sidebar.title("Navigation")

menu = st.sidebar.radio(
    "Select Page",
    [
        "Dashboard",
        "Analytics",
        "Prediction AI",
        "Diet Plan",
        "Advanced Diet AI", # Added Advanced Diet AI to the menu
        "Health Report"
    ]
)


# ------------------------------------------------------------
# Fitness Score Function
# ------------------------------------------------------------
def fitness_score(steps, sleep, bmi):
    score = 0

    if steps > 10000:
        score += 40
    elif steps > 7000:
        score += 30
    else:
        score += 15

    if sleep >= 7:
        score += 30
    else:
        score += 15

    if 18.5 <= bmi <= 24.9:
        score += 30
    else:
        score += 15

    return score

# ============================================================
# ADVANCED FITNESS INTELLIGENCE MODULE (FitnessAI class)
# ============================================================

class FitnessAI:

    def __init__(self):
        pass

    # --------------------------------------------------------
    # BMR Calculator
    # --------------------------------------------------------
    def calculate_bmr(self, weight, height, age, gender):

        if gender == "Male":
            bmr = 10 * weight + 6.25 * height - 5 * age + 5
        else:
            bmr = 10 * weight + 6.25 * height - 5 * age - 161

        return int(bmr)

    # --------------------------------------------------------
    # Daily Calories
    # --------------------------------------------------------
    def daily_calories(self, bmr, activity_level):

        activity_multiplier = {
            "Low": 1.2,
            "Moderate": 1.5,
            "High": 1.75
        }

        return int(bmr * activity_multiplier[activity_level])

    # --------------------------------------------------------
    # Macro Calculation
    # --------------------------------------------------------
    def macros(self, calories, goal):

        if goal == "Fat Loss":
            protein = calories * 0.40 / 4
            carbs = calories * 0.30 / 4
            fats = calories * 0.30 / 9

        elif goal == "Muscle Gain":
            protein = calories * 0.35 / 4
            carbs = calories * 0.45 / 4
            fats = calories * 0.20 / 9

        else:
            protein = calories * 0.30 / 4
            carbs = calories * 0.40 / 4
            fats = calories * 0.30 / 9

        return {
            "Protein (g)": int(protein),
            "Carbs (g)": int(carbs),
            "Fats (g)": int(fats)
        }

    # --------------------------------------------------------
    # Weekly Diet Plan
    # --------------------------------------------------------
    def weekly_diet(self):

        plan = {
            "Monday": "High Protein Meals",
            "Tuesday": "Balanced Diet",
            "Wednesday": "Low Carb Focus",
            "Thursday": "Muscle Recovery Meals",
            "Friday": "Protein Rich Diet",
            "Saturday": "Athlete Level Nutrition",
            "Sunday": "Light Detox Diet"
        }

        return plan

    # --------------------------------------------------------
    # AI Coach Advice
    # --------------------------------------------------------
    def ai_coach(self, fitness_level, sleep, steps):

        if fitness_level == "Underactive":
            return "Start walking daily and build consistency."

        if sleep < 6:
            return "Improve sleep for better recovery."

        if steps > 12000:
            return "Great work! Focus on recovery and hydration."

        return "Maintain balanced lifestyle."

# ------------------------------------------------------------
# DASHBOARD PAGE
# ------------------------------------------------------------
if menu == "Dashboard":

    st.subheader("Your Fitness Feed")

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.markdown('<div class="metric-box">📊 Records<br><br>'+str(len(df))+'</div>', unsafe_allow_html=True)

    with col2:
        st.markdown('<div class="metric-box">👣 Avg Steps<br><br>'+str(int(df["Steps"].mean()))+'</div>', unsafe_allow_html=True)

    with col3:
        st.markdown('<div class="metric-box">🔥 Calories<br><br>'+str(int(df["Calories"].mean()))+'</div>', unsafe_allow_html=True)

    with col4:
        st.markdown('<div class="metric-box">😴 Sleep<br><br>'+str(round(df["Sleep"].mean(),2))+'</div>', unsafe_allow_html=True)

    st.write("")

    st.markdown("### Activity Trends")

    fig, ax = plt.subplots()
    df["Fitness_Level"].value_counts().plot(kind="bar", ax=ax)
    st.pyplot(fig)


# ------------------------------------------------------------
# ANALYTICS PAGE
# ------------------------------------------------------------
elif menu == "Analytics":

    st.subheader("Dataset Overview")

    st.dataframe(df)

    st.subheader("Correlation Heatmap")

    corr = df.corr(numeric_only=True)

    fig, ax = plt.subplots()
    cax = ax.matshow(corr)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    st.pyplot(fig)

    st.subheader("Steps vs Calories")

    fig, ax = plt.subplots()
    ax.scatter(df["Steps"], df["Calories"])
    ax.set_xlabel("Steps")
    ax.set_ylabel("Calories")
    st.pyplot(fig)

# ------------------------------------------------------------
# AI PREDICTION PAGE
# ------------------------------------------------------------
elif menu == "Prediction AI":

    st.subheader("Check Your Fitness Status")

    col1, col2 = st.columns(2)

    with col1:
        steps = st.slider("Steps", 1000, 20000, 7000)
        calories = st.slider("Calories Burned", 1000, 4000, 2200)
        heart = st.slider("Heart Rate", 50, 160, 80)

    with col2:
        sleep = st.slider("Sleep Hours", 3.0, 10.0, 7.0)
        bmi = st.slider("BMI", 15.0, 35.0, 23.0)

    if st.button("Analyze My Fitness"):
        X = df.drop("Fitness_Level", axis=1)
        y = df["Fitness_Level"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        model = RandomForestClassifier(n_estimators=200)
        model.fit(X_train, y_train)

        user = pd.DataFrame({
            "Steps": [steps],
            "Calories": [calories],
            "Heart_Rate": [heart],
            "Sleep": [sleep],
            "BMI": [bmi]
        })

        prediction = model.predict(user)

        score = fitness_score(steps, sleep, bmi)

        st.success(f"🏆 Fitness Level: {prediction[0]}")
        st.info(f"⭐ Fitness Score: {score}/100")


# ------------------------------------------------------------
# AI DIET + DIET PLAN PAGE
# ------------------------------------------------------------
elif menu == "Diet Plan":

    st.subheader("🥗 AI Diet Planner")

    st.write("Generate a personalized diet plan using AI")

    # User Inputs
    col1, col2, col3 = st.columns(3)

    with col1:
        bmi = st.slider("BMI", 15.0, 35.0, 23.0)
        age = st.slider("Age", 15, 60, 22)

    with col2:
        steps = st.slider("Daily Steps", 1000, 20000, 8000)
        workout = st.slider("Workout (minutes)", 0, 120, 30)

    with col3:
        goal = st.selectbox(
            "Fitness Goal",
            ["Fat Loss", "Muscle Gain", "Maintain Fitness"]
        )

        diet_type = st.selectbox(
            "Diet Preference",
            ["Balanced", "High Protein", "Vegetarian", "Low Carb"]
        )

    generate = st.button("Generate AI Diet Plan")

    if generate:

        # --------------------------
        # Calorie Calculation
        # --------------------------
        calories = 1800
        calories += int(steps / 2000) * 50
        calories += int(workout / 30) * 100

        if goal == "Fat Loss":
            calories -= 300
        elif goal == "Muscle Gain":
            calories += 350

        # BMI Adjustment
        if bmi > 27:
            goal_result = "Fat Loss"
        elif bmi < 19:
            goal_result = "Weight Gain"
        else:
            goal_result = "Maintain Fitness"

        st.success(f"🎯 Goal: {goal_result}")
        st.info(f"🔥 Recommended Calories: {calories}")

        # --------------------------
        # Base Diet Plan
        # --------------------------
        breakfast = ["Oats", "Eggs", "Green Tea"]
        lunch = ["Brown Rice", "Chicken / Paneer", "Vegetables"]
        snack = ["Protein Shake", "Nuts"]
        dinner = ["Soup", "Salad", "Tofu / Fish"]

        # --------------------------
        # AI Diet Customization
        # --------------------------
        if diet_type == "Vegetarian":
            lunch = ["Paneer", "Quinoa", "Vegetables"]
            dinner = ["Tofu", "Salad", "Soup"]

        elif diet_type == "High Protein":
            breakfast = ["Egg Omelette", "Protein Shake"]
            snack = ["Greek Yogurt", "Almonds"]

        elif diet_type == "Low Carb":
            lunch = ["Grilled Chicken", "Broccoli"]
            dinner = ["Fish", "Vegetables"]

        # --------------------------
        # UI Layout
        # --------------------------
        st.markdown("### 🍽 Your Daily Meal Plan")

        col4, col5 = st.columns(2)

        with col4:
            st.markdown("### Breakfast")
            for f in breakfast:
                st.write("•",f)

            st.markdown("### Lunch")
            for f in lunch:
                st.write("•",f)

        with col5:
            st.markdown("### Snack")
            for f in snack:
                st.write("•",f)

            st.markdown("### Dinner")
            for f in dinner:
                st.write("•",f)

        # --------------------------
        # AI Nutrition Insights
        # --------------------------
        st.markdown("---")
        st.markdown("### 🧠 AI Nutrition Insights")

        if bmi > 27:
            st.warning("Your BMI suggests fat loss diet is recommended.")
        elif bmi < 19:
            st.warning("You may need a calorie surplus for healthy weight gain.")
        else:
            st.success("Your BMI is in a healthy range. Maintain balanced nutrition.")

        st.write("• Drink 3–4 liters of water")
        st.write("• Eat protein in every meal")
        st.write("• Avoid ultra-processed food")
        st.write("• Maintain consistent workout routine")


# ------------------------------------------------------------
# ADVANCED DIET AI PAGE
# ------------------------------------------------------------
elif menu == "Advanced Diet AI":

    st.header("Advanced Nutrition Intelligence")

    ai = FitnessAI()

    weight = st.number_input("Weight (kg)", 40, 120, 70)
    height = st.number_input("Height (cm)", 140, 210, 170)
    age = st.number_input("Age", 15, 70, 22)

    gender = st.selectbox("Gender", ["Male", "Female"])

    activity = st.selectbox(
        "Activity Level",
        ["Low", "Moderate", "High"]
    )

    goal = st.selectbox(
        "Fitness Goal",
        ["Fat Loss", "Muscle Gain", "Maintain"]
    )

    if st.button("Calculate Metrics"):
        bmr = ai.calculate_bmr(weight, height, age, gender)
        calories = ai.daily_calories(bmr, activity)
        macros = ai.macros(calories, goal)

        st.metric("BMR", bmr)
        st.metric("Daily Calories Needed", calories)

        st.subheader("Macro Distribution")

        for key, value in macros.items():
            st.write(f"{key}: {value}")

        st.subheader("Weekly Diet Plan")

        plan = ai.weekly_diet()

        for day, meal in plan.items():
            st.write(f"{day} → {meal}")

        # Assuming a default fitness level, sleep, steps for AI coach advice
        # In a real app, these would come from user inputs or prediction results
        ai_coach_advice = ai.ai_coach(fitness_level="Healthy", sleep=7.0, steps=8000)
        st.info(f"AI Coach Advice: {ai_coach_advice}")


# ------------------------------------------------------------
# HEALTH REPORT PAGE
# ------------------------------------------------------------
elif menu == "Health Report":

    st.subheader("AI Health Insights")

    st.write("Average Steps:", int(df["Steps"].mean()))
    st.write("Average Calories:", int(df["Calories"].mean()))
    st.write("Average Heart Rate:", int(df["Heart_Rate"].mean()))
    st.write("Average Sleep:", round(df["Sleep"].mean(),2))

    st.subheader("Recommended Lifestyle Plan")

    st.write("• Walk at least 8,000–10,000 steps daily")
    st.write("• Maintain balanced nutrition")
    st.write("• Sleep 7–8 hours")
    st.write("• Exercise regularly")

st.markdown("---")
st.write("Developed by Sarthak Pal | AI + Data Science Project")

Overwriting app.py


In [167]:
# Kill any existing Streamlit processes
!killall streamlit || true

In [168]:
# Restart Streamlit app in the background
!nohup streamlit run app.py &>/dev/null &

In [169]:
import time
from pyngrok import ngrok

time.sleep(5) # Give Streamlit a moment to start

# Terminate existing ngrok tunnels if any
ngrok.kill()

# Your ngrok auth token (replace if it's not already set in the environment or secrets)
# The token should be just the string starting with '2_' or 'NgrokClient_'
ngrok.set_auth_token("3BTNKBMGYVFFIleYMxF4gJ784Pt_tmnY4wPCAerg9QCT3Chq")

# Connect ngrok to the Streamlit port
public_url = ngrok.connect(addr='8501', proto='http')
print(f'Your Streamlit app is live at: {public_url}')

Your Streamlit app is live at: NgrokTunnel: "https://nonphrenetically-demagogic-rodolfo.ngrok-free.dev" -> "http://localhost:8501"


In [170]:
!nohup streamlit run app.py &>/dev/null &
import time
time.sleep(3)
from pyngrok import ngrok

# --- IMPORTANT: Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual ngrok authtoken ---
# The token should be just the string starting with '2_' or 'NgrokClient_'
ngrok.set_auth_token("3BTNKBMGYVFFIleYMxF4gJ784Pt_tmnY4wPCAerg9QCT3Chq")

public_url = ngrok.connect(addr='8501', proto='http')
print(f'Your Streamlit app is live at: {public_url}')

Your Streamlit app is live at: NgrokTunnel: "https://nonphrenetically-demagogic-rodolfo.ngrok-free.dev" -> "http://localhost:8501"


In [171]:
# Kill any existing Streamlit processes
!killall streamlit || true

In [172]:
# Restart Streamlit app in the background
!nohup streamlit run app.py &>/dev/null &

In [173]:
import time
from pyngrok import ngrok

time.sleep(10) # Increased sleep to give Streamlit more time to start

# Terminate existing ngrok tunnels if any
ngrok.kill()

# Your ngrok auth token (replace if it's not already set in the environment or secrets)
# The token should be just the string starting with '2_' or 'NgrokClient_'
ngrok.set_auth_token("3BTNKBMGYVFFIleYMxF4gJ784Pt_tmnY4wPCAerg9QCT3Chq")

# Connect ngrok to the Streamlit port
public_url = ngrok.connect(addr='8501', proto='http')
print(f'Your Streamlit app is live at: {public_url}')

Your Streamlit app is live at: NgrokTunnel: "https://nonphrenetically-demagogic-rodolfo.ngrok-free.dev" -> "http://localhost:8501"


In [174]:
# Kill any existing Streamlit processes
!killall streamlit || true

In [175]:
# Restart Streamlit app in the background
!nohup streamlit run app.py &>/dev/null &

In [176]:
import time
from pyngrok import ngrok

time.sleep(15) # Increased sleep to give Streamlit more time to start

# Terminate existing ngrok tunnels if any
ngrok.kill()

# Your ngrok auth token (replace if it's not already set in the environment or secrets)
# The token should be just the string starting with '2_' or 'NgrokClient_'
ngrok.set_auth_token("3BTNKBMGYVFFIleYMxF4gJ784Pt_tmnY4wPCAerg9QCT3Chq")

# Connect ngrok to the Streamlit port
public_url = ngrok.connect(addr='8501', proto='http')
print(f'Your Streamlit app is live at: {public_url}')

Your Streamlit app is live at: NgrokTunnel: "https://nonphrenetically-demagogic-rodolfo.ngrok-free.dev" -> "http://localhost:8501"


In [177]:
!nohup streamlit run app.py &>/dev/null &
import time
time.sleep(3)
from pyngrok import ngrok

# The token should be just the string starting with '2_' or 'NgrokClient_'
ngrok.set_auth_token("3BTNKBMGYVFFIleYMxF4gJ784Pt_tmnY4wPCAerg9QCT3Chq")

public_url = ngrok.connect(addr='8501', proto='http')
print(f'Your Streamlit app is live at: {public_url}')

Your Streamlit app is live at: NgrokTunnel: "https://nonphrenetically-demagogic-rodolfo.ngrok-free.dev" -> "http://localhost:8501"


In [178]:
# Kill any existing Streamlit processes
!killall streamlit || true

# Restart Streamlit app in the background
!nohup streamlit run app.py &>/dev/null &

In [179]:
import time
from pyngrok import ngrok

time.sleep(5) # Give Streamlit a moment to start

# Terminate existing ngrok tunnels if any
ngrok.kill()

# Your ngrok auth token (replace if it's not already set in the environment or secrets)
# The token should be just the string starting with '2_' or 'NgrokClient_'
ngrok.set_auth_token("3BTNKBMGYVFFIleYMxF4gJ784Pt_tmnY4wPCAerg9QCT3Chq")

# Connect ngrok to the Streamlit port
public_url = ngrok.connect(addr='8501', proto='http')
print(f'Your Streamlit app is live at: {public_url}')

Your Streamlit app is live at: NgrokTunnel: "https://nonphrenetically-demagogic-rodolfo.ngrok-free.dev" -> "http://localhost:8501"


In [180]:
# ============================================================
# DIET RECOMMENDATION SYSTEM
# ============================================================

def generate_diet_plan(fitness_level, bmi, steps, workout):

    diet = {}

    # Base calorie estimation
    base_calories = 1800

    if fitness_level == "Underactive":
        calories = base_calories - 200
    elif fitness_level == "Moderate":
        calories = base_calories
    elif fitness_level == "Healthy":
        calories = base_calories + 200
    else:
        calories = base_calories + 400

    # Adjust based on activity
    calories += int(steps / 2000) * 50
    calories += int(workout / 30) * 100

    # BMI adjustments
    if bmi > 27:
        goal = "Fat Loss"
        calories -= 300
    elif bmi < 19:
        goal = "Weight Gain"
        calories += 300
    else:
        goal = "Maintain Fitness"

    # ============================================================
# DIET RECOMMENDATION SYSTEM (BMI-BASED SMART DIET ENGINE)
# ============================================================

def generate_diet_plan(fitness_level, bmi, steps, workout):

    diet = {}

    # Base calorie estimation
    base_calories = 1800

    if fitness_level == "Underactive":
        calories = base_calories - 200
    elif fitness_level == "Moderate":
        calories = base_calories
    elif fitness_level == "Healthy":
        calories = base_calories + 200
    else:
        calories = base_calories + 400

    # Adjust calories based on activity
    calories += int(steps / 2000) * 50
    calories += int(workout / 30) * 100

    # =========================================================
    # BMI-BASED GOAL + DIET PLAN
    # =========================================================

    # UNDERWEIGHT
    if bmi < 18.5:
        goal = "Weight Gain"
        calories += 400

        breakfast = [
            "Peanut butter toast",
            "Banana smoothie",
            "Oatmeal with milk",
            "Boiled eggs / Paneer bhurji"
        ]

        lunch = [
            "Rice + dal",
            "Chicken / paneer curry",
            "Avocado or healthy fats",
            "Yogurt"
        ]

        snack = [
            "Protein shake",
            "Almonds & cashews",
            "Peanut chikki"
        ]

        dinner = [
            "Whole wheat roti",
            "Paneer / fish",
            "Vegetables",
            "Milk before sleep"
        ]

    # NORMAL BMI
    elif 18.5 <= bmi <= 24.9:
        goal = "Maintain Fitness"

        breakfast = [
            "Oatmeal with fruits",
            "Boiled eggs / Paneer",
            "Green tea",
            "Whole grain toast"
        ]

        lunch = [
            "Brown rice or roti",
            "Grilled chicken / dal",
            "Vegetables",
            "Salad"
        ]

        snack = [
            "Fruit bowl",
            "Nuts",
            "Protein shake"
        ]

        dinner = [
            "Light meal",
            "Soup + vegetables",
            "Tofu / paneer / fish"
        ]

    # OVERWEIGHT
    elif 25 <= bmi <= 29.9:
        goal = "Fat Loss"
        calories -= 300

        breakfast = [
            "Boiled eggs",
            "Black coffee / green tea",
            "Low carb oats",
            "Fruit (apple)"
        ]

        lunch = [
            "Grilled chicken / tofu",
            "Large salad bowl",
            "Low carb roti",
            "Vegetables"
        ]

        snack = [
            "Greek yogurt",
            "Handful almonds",
            "Protein shake"
        ]

        dinner = [
            "Vegetable soup",
            "Grilled paneer / chicken",
            "Steamed vegetables"
        ]

    # OBESE
    else:
        goal = "Aggressive Fat Loss"
        calories -= 500

        breakfast = [
            "Green smoothie",
            "Boiled eggs",
            "Chia seed pudding"
        ]

        lunch = [
            "Large salad",
            "Lean protein (fish/chicken/tofu)",
            "Vegetables"
        ]

        snack = [
            "Green tea",
            "Walnuts",
            "Low fat yogurt"
        ]

        dinner = [
            "Light soup",
            "Steamed vegetables",
            "Small protein portion"
        ]

    # =========================================================
    # FINAL DIET STRUCTURE
    # =========================================================

    diet["Goal"] = goal
    diet["Recommended Calories"] = int(calories)
    diet["Breakfast"] = breakfast
    diet["Lunch"] = lunch
    diet["Snack"] = snack
    diet["Dinner"] = dinner

    return diet


In [181]:
page = st.sidebar.radio(
    "Select Module",
    [
        "Executive Dashboard",
        "AI Prediction Engine",
        "Diet Planner AI",
        "Advanced Analytics",
        "Model Intelligence",
        "Health Report Generator"
    ]
)


2026-03-27 11:13:01.415 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.416 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.419 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.420 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.421 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.424 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [182]:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Assuming load_data and generate_diet_plan are defined as in app.py
# and that df, model are available from app.py context

# --- Placeholder for df and model for isolated execution demonstration ---
# In the actual app.py, df and model are defined globally or via @st.cache_data / @st.cache_resource

# Re-create load_data to get df
@st.cache_data
def load_data_for_demo():
    np.random.seed(42)
    data = pd.DataFrame({
        "Steps": np.random.randint(2000, 15000, 500),
        "Calories": np.random.randint(1500, 3500, 500),
        "Heart_Rate": np.random.randint(60, 140, 500),
        "Sleep": np.random.uniform(4, 9, 500),
        "BMI": np.random.uniform(18, 32, 500)
    })
    conditions = [
        (data["Steps"] < 4000),
        (data["Steps"] < 8000),
        (data["Steps"] < 12000),
        (data["Steps"] >= 12000)
    ]
    labels = ["Underactive", "Moderate", "Healthy", "Highly Active"]
    data["Fitness_Level"] = np.select(conditions, labels, default='Unknown')
    return data

df_demo = load_data_for_demo()

# Re-create generate_diet_plan for isolated execution demonstration
def generate_diet_plan_for_demo(fitness_level, bmi, steps, workout):
    diet = {}
    base_calories = 1800
    if fitness_level == "Underactive":
        calories = base_calories - 200
    elif fitness_level == "Moderate":
        calories = base_calories
    elif fitness_level == "Healthy":
        calories = base_calories + 200
    else:
        calories = base_calories + 400
    calories += int(steps / 2000) * 50
    calories += int(workout / 30) * 100
    if bmi > 27:
        goal = "Fat Loss"
        calories -= 300
    elif bmi < 19:
        goal = "Weight Gain"
        calories += 300
    else:
        goal = "Maintain Fitness"
    diet["Goal"] = goal
    diet["Recommended Calories"] = calories
    diet["Breakfast"] = ["Oatmeal with fruits", "Boiled eggs / Paneer", "Green tea"]
    diet["Lunch"] = ["Brown rice or roti", "Grilled chicken / dal", "Vegetables", "Salad"]
    diet["Snack"] = ["Protein shake", "Nuts", "Banana"]
    diet["Dinner"] = ["Light meal", "Soup + vegetables", "Paneer / tofu / fish"]
    return diet

# Train a model for demo purposes
X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    df_demo.drop("Fitness_Level", axis=1), df_demo["Fitness_Level"], test_size=0.2, random_state=42
)
model_demo = RandomForestClassifier()
model_demo.fit(X_train_demo, y_train_demo)



# Corrected 'Diet Planner AI' section (extracted from app.py)
# Note: In a live Streamlit app, 'menu' would be controlled by st.sidebar.radio
# and df/model would be globally available or cached.

# For demonstration, let's assume 'menu' is 'Diet Planner AI' for this block to be active.
# In a real Streamlit app, this would be within the main `if/elif` structure.

st.header("🥗 AI Diet Recommendation System")
st.write("Personalized nutrition plan based on your fitness results")

col1, col2 = st.columns(2)

with col1:
    steps_input = st.slider("Daily Steps", 1000, 20000, 8000, key="diet_steps_demo")
    bmi_input = st.slider("BMI", 15.0, 38.0, 23.0, key="diet_bmi_demo")

with col2:
    workout_input = st.slider("Workout Minutes", 0, 180, 40, key="diet_workout_demo")
    sleep_input = st.slider("Sleep Hours", 3.0, 10.0, 7.0, key="diet_sleep_demo")

user_for_prediction_demo = pd.DataFrame({
    "Steps": [steps_input],
    "Calories": [df_demo['Calories'].mean()],
    "Heart_Rate": [df_demo['Heart_Rate'].mean()],
    "Sleep": [sleep_input],
    "BMI": [bmi_input]
})

fitness_level_demo = model_demo.predict(user_for_prediction_demo)[0]

st.success(f"Detected Fitness Level: {fitness_level_demo}")

diet_demo = generate_diet_plan_for_demo(fitness_level_demo, bmi_input, steps_input, workout_input)

st.subheader("Your Goal")
st.info(diet_demo["Goal"])

st.subheader("Daily Calories Needed")
st.metric("Calories", diet_demo["Recommended Calories"])

st.subheader("🍳 Breakfast")
for item in diet_demo["Breakfast"]:
    st.write("•", item)

st.subheader("🍛 Lunch")
for item in diet_demo["Lunch"]:
    st.write("•", item)

st.subheader("🥤 Snacks")
for item in diet_demo["Snack"]:
    st.write("•", item)

st.subheader("🍲 Dinner")
for item in diet_demo["Dinner"]:
    st.write("•", item)


2026-03-27 11:13:01.451 No runtime found, using MemoryCacheStorageManager
2026-03-27 11:13:01.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.677 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.679 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.680 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:01.681 Thread 'MainThread':

In [183]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Re-define the load_data function from app.py to generate the dataset
# This ensures the model is trained on the same data distribution as the app.
def load_data():
    np.random.seed(42)

    data = pd.DataFrame({
        "Steps": np.random.randint(2000, 15000, 500),
        "Calories": np.random.randint(1500, 3500, 500),
        "Heart_Rate": np.random.randint(60, 140, 500),
        "Sleep": np.random.uniform(4, 9, 500),
        "BMI": np.random.uniform(18, 32, 500)
    })

    conditions = [
        (data["Steps"] < 4000),
        (data["Steps"] < 8000),
        (data["Steps"] < 12000),
        (data["Steps"] >= 12000)
    ]

    labels = ["Underactive", "Moderate", "Healthy", "Highly Active"]
    data["Fitness_Level"] = np.select(conditions, labels, default='Unknown')

    return data

df = load_data()

# Train the model as done in app.py
X = df.drop("Fitness_Level", axis=1)
y = df["Fitness_Level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

# Sample input data for prediction
sample_steps = 7500
sample_calories = 2500
sample_heart_rate = 75
sample_sleep = 7.5
sample_bmi = 24.0

user_input = pd.DataFrame({
    "Steps": [sample_steps],
    "Calories": [sample_calories],
    "Heart_Rate": [sample_heart_rate],
    "Sleep": [sample_sleep],
    "BMI": [sample_bmi]
})

# Make a prediction
predicted_fitness_level = model.predict(user_input)

print(f"Sample Input: Steps={sample_steps}, Calories={sample_calories}, Heart Rate={sample_heart_rate}, Sleep={sample_sleep}, BMI={sample_bmi}")
print(f"Predicted Fitness Level: {predicted_fitness_level[0]}")

Sample Input: Steps=7500, Calories=2500, Heart Rate=75, Sleep=7.5, BMI=24.0
Predicted Fitness Level: Moderate


In [184]:
!netstat -tuln | grep 8501

tcp        0      0 0.0.0.0:8501            0.0.0.0:*               LISTEN     
tcp6       0      0 :::8501                 :::*                    LISTEN     


In [185]:
# ============================================================
# FITNESS ACTIVITY ANALYSIS USING MACHINE LEARNING
# Streamlit + ML Project
# By: Sarthak Pal
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# Page Configuration
# ------------------------------------------------------------
st.set_page_config(page_title="Fitness Activity Dashboard", layout="wide")

st.title("🏃 Fitness Activity Analysis using Machine Learning")
st.subheader("Metasatellite Health Intelligence System")

# ------------------------------------------------------------
# Create Built-in Dataset
# ------------------------------------------------------------
@st.cache_data
def load_data():
    np.random.seed(42)

    data = pd.DataFrame({
        "Steps": np.random.randint(2000, 15000, 300),
        "Calories_Burned": np.random.randint(1500, 3500, 300),
        "Heart_Rate": np.random.randint(60, 140, 300),
        "Sleep_Hours": np.random.uniform(4, 9, 300),
        "BMI": np.random.uniform(18, 32, 300)
    })

    # Fitness Level Logic
    conditions = [
        (data["Steps"] < 4000),
        (data["Steps"] >= 4000) & (data["Steps"] < 8000),
        (data["Steps"] >= 8000) & (data["Steps"] < 12000),
        (data["Steps"] >= 12000)
    ]

    labels = ["Underactive", "Moderate", "Healthy", "Highly Active"]

    data["Fitness_Level"] = np.select(conditions, labels, default='Unknown')

    return data


df = load_data()

# ------------------------------------------------------------
# Sidebar
# ------------------------------------------------------------
st.sidebar.header("Enter Your Daily Activity")

steps = st.sidebar.slider("Daily Steps", 1000, 20000, 6000)
calories = st.sidebar.slider("Calories Burned", 1000, 4000, 2200)
heart_rate = st.sidebar.slider("Heart Rate", 50, 160, 80)
sleep = st.sidebar.slider("Sleep Hours", 3.0, 10.0, 7.0)
bmi = st.sidebar.slider("BMI", 15.0, 35.0, 23.0)

user_data = pd.DataFrame({
    "Steps": [steps],
    "Calories_Burned": [calories],
    "Heart_Rate": [heart_rate],
    "Sleep_Hours": [sleep],
    "BMI": [bmi]
})

# ------------------------------------------------------------
# Dataset Preview
# ------------------------------------------------------------
st.subheader("Dataset Preview")
st.dataframe(df.head())

# ------------------------------------------------------------
# Basic Dataset Info
# ------------------------------------------------------------
st.subheader("Dataset Summary")

col1, col2, col3 = st.columns(3)

col1.metric("Total Records", len(df))
col2.metric("Average Steps", int(df["Steps"].mean()))
col3.metric("Average Sleep Hours", round(df["Sleep_Hours"].mean(), 2))

# ------------------------------------------------------------
# Exploratory Data Analysis
# ------------------------------------------------------------
st.subheader("Exploratory Data Analysis")

col1, col2 = st.columns(2)

with col1:
    fig1, ax1 = plt.subplots()
    ax1.hist(df["Steps"], bins=20)
    ax1.set_title("Steps Distribution")
    ax1.grid(True, linestyle='--', alpha=0.6)
    fig1.tight_layout()
    st.pyplot(fig1)

with col2:
    fig2, ax2 = plt.subplots()
    ax2.hist(df["Calories_Burned"], bins=20)
    ax2.set_title("Calories Burned Distribution")
    ax2.grid(True, linestyle='--', alpha=0.6)
    fig2.tight_layout()
    st.pyplot(fig2)

col3, col4 = st.columns(2)

with col3:
    fig3, ax3 = plt.subplots()
    ax3.scatter(df["Steps"], df["Calories_Burned"])
    ax3.set_xlabel("Steps")
    ax3.set_ylabel("Calories Burned")
    ax3.set_title("Steps vs Calories")
    ax3.grid(True, linestyle='--', alpha=0.6)
    fig3.tight_layout()
    st.pyplot(fig3)

with col4:
    fig4, ax4 = plt.subplots()
    ax4.hist(df["Sleep_Hours"], bins=20)
    ax4.set_title("Sleep Hours Distribution")
    ax4.grid(True, linestyle='--', alpha=0.6)
    fig4.tight_layout()
    st.pyplot(fig4)

# ------------------------------------------------------------
# Machine Learning Model
# ------------------------------------------------------------
st.subheader("Machine Learning Model Training")

X = df.drop("Fitness_Level", axis=1)
y = df["Fitness_Level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

st.success(f"Model Accuracy: {round(accuracy * 100, 2)} %")

# ------------------------------------------------------------
# Feature Importance
# ------------------------------------------------------------
st.subheader("Feature Importance")

importance = model.feature_importances_
features = X.columns

fig5, ax5 = plt.subplots()
ax5.barh(features, importance)
ax5.set_title("Feature Importance in Prediction")
fig5.tight_layout()
st.pyplot(fig5)

# ------------------------------------------------------------
# User Prediction
# ------------------------------------------------------------
st.subheader("Your Fitness Prediction")

prediction = model.predict(user_data)

st.success(f"Predicted Fitness Level: {prediction[0]}")

# ------------------------------------------------------------
# Recommendations
# ------------------------------------------------------------
st.subheader("Personalized Health Recommendations")

if prediction[0] == "Underactive":
    st.warning("You need more physical activity.")
    st.write("Recommended Plan:")
    st.write("• Walk 8,000–10,000 steps daily")
    st.write("• Start light cardio workouts")
    st.write("• Eat more protein and vegetables")

elif prediction[0] == "Moderate":
    st.info("You are moderately active.")
    st.write("Recommended Plan:")
    st.write("• Increase workouts to 4 days per week")
    st.write("• Maintain balanced diet")
    st.write("• Improve sleep quality")

elif prediction[0] == "Healthy":
    st.success("Great! You are maintaining a healthy lifestyle.")
    st.write("Recommended Plan:")
    st.write("• Continue regular workouts")
    st.write("• Add strength training")
    st.write("• Maintain nutrition balance")

elif prediction[0] == "Highly Active":
    st.success("Excellent fitness level!")
    st.write("Recommended Plan:")
    st.write("• Maintain workout consistency")
    st.write("• High protein diet")
    st.write("• Proper recovery and hydration")

# ------------------------------------------------------------
# Health Report
# ------------------------------------------------------------
st.subheader("Health Insight Report")

report = pd.DataFrame({
    "Metric": ["Steps", "Calories Burned", "Heart Rate", "Sleep Hours", "BMI"],
    "Your Value": [steps, calories, heart_rate, sleep, bmi]
})

st.table(report)

st.write("Overall Fitness Status:", prediction[0])

st.markdown("---")
st.write("Developed by **Sarthak Pal** | Data Science Project")

2026-03-27 11:13:02.252 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:02.253 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:02.255 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:02.256 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:02.258 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:02.259 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:02.260 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:02.261 No runtime found, using MemoryCacheStorageManager
2026-03-27 11:13:02.265 Thread 'MainThread':

In [186]:
# ============================================================
# FITNESS ACTIVITY ANALYSIS DASHBOARD (ADVANCED UI)
# Modern Streamlit Dashboard
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# Page Configuration
# ------------------------------------------------------------
st.set_page_config(
    page_title="Fitness AI Dashboard",
    page_icon="🏋️",
    layout="wide"
)

# ------------------------------------------------------------
# Custom UI Styling
# ------------------------------------------------------------
st.markdown("""
<style>
.big-title {
    font-size:40px;
    font-weight:bold;
    color:#00C2FF;
}
.card {
    padding:20px;
    border-radius:15px;
    background-color:#111111;
    color:white;
    box-shadow:0px 4px 10px rgba(0,0,0,0.3);
}
.metric {
    font-size:25px;
    font-weight:bold;
}
</style>
""", unsafe_allow_html=True)

st.markdown('<p class="big-title">🏃 Fitness Activity AI Dashboard</p>', unsafe_allow_html=True)
st.write("Machine Learning Powered Health Insights")

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
@st.cache_data
def load_data():
    np.random.seed(42)
    data = pd.DataFrame({
        "Steps": np.random.randint(2000, 15000, 400),
        "Calories": np.random.randint(1500, 3500, 400),
        "Heart_Rate": np.random.randint(60, 140, 400),
        "Sleep": np.random.uniform(4, 9, 400),
        "BMI": np.random.uniform(18, 32, 400)
    })

    conditions = [
        (data["Steps"] < 4000),
        (data["Steps"] < 8000),
        (data["Steps"] < 12000),
        (data["Steps"] >= 12000)
    ]

    labels = ["Underactive", "Moderate", "Healthy", "Highly Active"]
    data["Fitness_Level"] = np.select(conditions, labels, default='Unknown')

    return data

df = load_data()

# ------------------------------------------------------------
# Sidebar Navigation
# ------------------------------------------------------------
st.sidebar.title("Dashboard Menu")

menu = st.sidebar.radio(
    "Navigate",
    ["Dashboard", "Data Analysis", "Fitness Prediction", "Health Report"]
)

# ------------------------------------------------------------
# DASHBOARD
# ------------------------------------------------------------
if menu == "Dashboard":

    st.subheader("Fitness Overview")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Users", len(df))
    col2.metric("Avg Steps", int(df["Steps"].mean()))
    col3.metric("Avg Calories", int(df["Calories"].mean()))
    col4.metric("Avg Sleep", round(df["Sleep"].mean(), 2))

    st.subheader("Activity Trends")

    col1, col2 = st.columns(2)

    with col1:
        fig, ax = plt.subplots()
        ax.hist(df["Steps"], bins=25)
        ax.set_title("Daily Steps Distribution")
        st.pyplot(fig)

    with col2:
        fig, ax = plt.subplots()
        ax.hist(df["Calories"], bins=25)
        ax.set_title("Calories Burned")
        st.pyplot(fig)

    st.subheader("Fitness Category Distribution")

    fig, ax = plt.subplots()
    df["Fitness_Level"].value_counts().plot(kind="bar", ax=ax)
    st.pyplot(fig)

# ------------------------------------------------------------
# DATA ANALYSIS
# ------------------------------------------------------------
elif menu == "Data Analysis":

    st.subheader("Dataset")

    st.dataframe(df)

    st.subheader("Correlation Analysis")

    corr = df.corr(numeric_only=True)

    fig, ax = plt.subplots()
    cax = ax.matshow(corr)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    st.pyplot(fig)

# ------------------------------------------------------------
# MACHINE LEARNING
# ------------------------------------------------------------
elif menu == "Fitness Prediction":

    st.subheader("Enter Your Daily Activity")

    steps = st.slider("Steps", 1000, 20000, 7000)
    calories = st.slider("Calories Burned", 1000, 4000, 2200)
    heart = st.slider("Heart Rate", 50, 160, 80)
    sleep = st.slider("Sleep Hours", 3.0, 10.0, 7.0)
    bmi = st.slider("BMI", 15.0, 35.0, 23.0)

    X = df.drop("Fitness_Level", axis=1)
    y = df["Fitness_Level"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    user = pd.DataFrame({
        "Steps": [steps],
        "Calories": [calories],
        "Heart_Rate": [heart],
        "Sleep": [sleep],
        "BMI": [bmi]
    })

    prediction = model.predict(user)

    st.success(f"Your Fitness Level: {prediction[0]}")

# ------------------------------------------------------------
# HEALTH REPORT
# ------------------------------------------------------------
elif menu == "Health Report":

    st.subheader("Generated Health Insights")

    st.write("This report summarizes the user's fitness performance.")

    st.write("Average Steps:", int(df["Steps"].mean()))
    st.write("Average Calories:", int(df["Calories"].mean()))
    st.write("Average Heart Rate:", int(df["Heart_Rate"].mean()))
    st.write("Average Sleep Hours:", round(df["Sleep"].mean(), 2))

    st.subheader("Recommendations")

    st.write("• Maintain regular exercise")
    st.write("• Sleep 7–8 hours daily")
    st.write("• Balanced diet")
    st.write("• Track your activity daily")

st.markdown("---")
st.write("Developed by Sarthak Pal | Machine Learning Project")

2026-03-27 11:13:04.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:04.127 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:04.128 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:04.129 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:04.134 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:04.135 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:04.138 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-27 11:13:04.140 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [187]:
# ============================================================
# ADVANCED FITNESS INTELLIGENCE MODULE
# ============================================================

class FitnessAI:

    def __init__(self):
        pass

    # --------------------------------------------------------
    # BMR Calculator
    # --------------------------------------------------------
    def calculate_bmr(self, weight, height, age, gender):

        if gender == "Male":
            bmr = 10 * weight + 6.25 * height - 5 * age + 5
        else:
            bmr = 10 * weight + 6.25 * height - 5 * age - 161

        return int(bmr)

    # --------------------------------------------------------
    # Daily Calories
    # --------------------------------------------------------
    def daily_calories(self, bmr, activity_level):

        activity_multiplier = {
            "Low": 1.2,
            "Moderate": 1.5,
            "High": 1.75
        }

        return int(bmr * activity_multiplier[activity_level])

    # --------------------------------------------------------
    # Macro Calculation
    # --------------------------------------------------------
    def macros(self, calories, goal):

        if goal == "Fat Loss":
            protein = calories * 0.40 / 4
            carbs = calories * 0.30 / 4
            fats = calories * 0.30 / 9

        elif goal == "Muscle Gain":
            protein = calories * 0.35 / 4
            carbs = calories * 0.45 / 4
            fats = calories * 0.20 / 9

        else:
            protein = calories * 0.30 / 4
            carbs = calories * 0.40 / 4
            fats = calories * 0.30 / 9

        return {
            "Protein (g)": int(protein),
            "Carbs (g)": int(carbs),
            "Fats (g)": int(fats)
        }

    # --------------------------------------------------------
    # Weekly Diet Plan
    # --------------------------------------------------------
    def weekly_diet(self):

        plan = {
            "Monday": "High Protein Meals",
            "Tuesday": "Balanced Diet",
            "Wednesday": "Low Carb Focus",
            "Thursday": "Muscle Recovery Meals",
            "Friday": "Protein Rich Diet",
            "Saturday": "Athlete Level Nutrition",
            "Sunday": "Light Detox Diet"
        }

        return plan

    # --------------------------------------------------------
    # AI Coach Advice
    # --------------------------------------------------------
    def ai_coach(self, fitness_level, sleep, steps):

        if fitness_level == "Underactive":
            return "Start walking daily and build consistency."

        if sleep < 6:
            return "Improve sleep for better recovery."

        if steps > 12000:
            return "Great work! Focus on recovery and hydration."

        return "Maintain balanced lifestyle."


In [188]:
# ============================================================
# FOOD + FITNESS DATASET GENERATOR FOR STREAMLIT APP
# Run this once in Google Colab
# It will create a dataset used by your Diet AI
# ============================================================

import pandas as pd
import numpy as np

print("Creating Nutrition Dataset...")

# ============================================================
# LARGE FOOD DATABASE
# ============================================================

foods = [
    "Oats","Eggs","Chicken Breast","Brown Rice","Paneer","Tofu",
    "Banana","Apple","Almonds","Milk","Yogurt","Fish",
    "Quinoa","Sweet Potato","Broccoli","Spinach","Avocado",
    "Peanut Butter","Lentils","Chickpeas","Orange","Mango",
    "Walnuts","Cashews","Cottage Cheese","Turkey","Salmon",
    "Whole Wheat Bread","Pasta","Greek Yogurt","Pumpkin Seeds",
    "Chia Seeds","Dark Chocolate","Strawberries","Blueberries",
    "Cucumber","Tomato","Carrot","Mushroom","Egg Whites"
]

categories = [
    "Healthy","Protein","Protein","Carbs","Protein","Protein",
    "Fruit","Fruit","Healthy Fat","Dairy","Dairy","Protein",
    "Healthy","Carbs","Vegetable","Vegetable","Healthy Fat",
    "Healthy Fat","Protein","Protein","Fruit","Fruit",
    "Healthy Fat","Healthy Fat","Dairy","Protein","Protein",
    "Carbs","Carbs","Dairy","Healthy Fat","Healthy Fat",
    "Healthy","Fruit","Fruit","Vegetable","Vegetable",
    "Vegetable","Vegetable","Protein"
]

np.random.seed(42)

food_dataset = pd.DataFrame({
    "Food": foods,
    "Category": categories,
    "Calories": np.random.randint(50, 600, len(foods)),
    "Protein": np.random.randint(1, 40, len(foods)),
    "Carbs": np.random.randint(1, 80, len(foods)),
    "Fat": np.random.randint(1, 30, len(foods))
})

# ============================================================
# ADD FITNESS RELATED FOODS
# ============================================================

fitness_foods = pd.DataFrame({
    "Food": [
        "Protein Shake","Energy Bar","Electrolyte Drink",
        "Sports Drink","Protein Pancakes","Muscle Gain Smoothie"
    ],
    "Category": ["Protein","Energy","Recovery","Energy","Protein","Protein"],
    "Calories": [200,250,120,150,300,350],
    "Protein": [30,10,0,0,25,35],
    "Carbs": [10,40,15,35,20,25],
    "Fat": [2,5,0,0,6,8]
})

food_dataset = pd.concat([food_dataset, fitness_foods], ignore_index=True)

# ============================================================
# SAVE DATASET
# ============================================================

food_dataset.to_csv("food_dataset.csv", index=False)

print("Dataset saved as food_dataset.csv")

# ============================================================
# PREVIEW DATA
# ============================================================

print("\nDataset Preview:")
print(food_dataset.head())

print("\nTotal Foods in Dataset:", len(food_dataset))

# ============================================================
# CODE YOU WILL ADD IN YOUR STREAMLIT APP
# ============================================================

print("\nAdd this inside your Streamlit app.py:\n")

print("""
@st.cache_data
def load_food_dataset():
    return pd.read_csv("food_dataset.csv")

food_db = load_food_dataset()
""")


Creating Nutrition Dataset...
Dataset saved as food_dataset.csv

Dataset Preview:
             Food Category  Calories  Protein  Carbs  Fat
0            Oats  Healthy       152       25      9   29
1            Eggs  Protein       485       14      1    4
2  Chicken Breast  Protein       320        9      8    5
3      Brown Rice    Carbs       156       26     63   23
4          Paneer  Protein       121        2     11    7

Total Foods in Dataset: 46

Add this inside your Streamlit app.py:


@st.cache_data
def load_food_dataset():
    return pd.read_csv("food_dataset.csv")

food_db = load_food_dataset()



In [189]:
# ============================================================
# DOWNLOAD REAL FOOD & FITNESS DATASETS
# ============================================================

import pandas as pd

# Food Nutrition Dataset (based on USDA style datasets used in research)
food_url = "https://raw.githubusercontent.com/mledoze/countries/master/data/countries.csv"

# Nutrition dataset alternative (food calories dataset used in ML examples)
nutrition_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"

# Fitness activity dataset
fitness_url = "https://raw.githubusercontent.com/plotly/datasets/master/2016-weather-data-seattle.csv"

try:
    food_df = pd.read_csv(food_url)
    nutrition_df = pd.read_csv(nutrition_url)
    fitness_df = pd.read_csv(fitness_url)

    print("Datasets Loaded Successfully")

    print("Food Dataset Shape:", food_df.shape)
    print("Nutrition Dataset Shape:", nutrition_df.shape)
    print("Fitness Dataset Shape:", fitness_df.shape)

except Exception as e:
    print("Error loading datasets:", e)


Error loading datasets: HTTP Error 404: Not Found


In [190]:
nutrition_data_url = "https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-population.csv"

nutrition_df = pd.read_csv(nutrition_data_url)

nutrition_df.head()


,state/region,ages,year,population
0,AL,under18,2012,1117489.0
1,AL,total,2012,4817528.0
2,AL,under18,2010,1130966.0
3,AL,total,2010,4785570.0
4,AL,under18,2011,1125763.0


In [191]:
# ============================================================
# CREATE FOOD DATABASE FOR DIET AI
# ============================================================

food_database = pd.DataFrame({
    "Food": [
        "Oats", "Eggs", "Chicken Breast", "Brown Rice",
        "Paneer", "Tofu", "Banana", "Apple", "Almonds",
        "Milk", "Yogurt", "Fish", "Quinoa", "Sweet Potato"
    ],
    "Calories": [
        389, 155, 165, 216,
        265, 144, 89, 52, 579,
        103, 59, 206, 120, 86
    ],
    "Protein": [
        17, 13, 31, 5,
        18, 15, 1, 0, 21,
        8, 10, 22, 4, 2
    ],
    "Carbs": [
        66, 1, 0, 45,
        6, 3, 23, 14, 22,
        12, 3, 0, 21, 20
    ],
    "Category": [
        "Healthy", "Protein", "Protein", "Carbs",
        "Protein", "Protein", "Fruit", "Fruit", "Healthy Fat",
        "Dairy", "Dairy", "Protein", "Healthy", "Carbs"
    ]
})

food_database.head()


,Food,Calories,Protein,Carbs,Category
0,Oats,389,17,66,Healthy
1,Eggs,155,13,1,Protein
2,Chicken Breast,165,31,0,Protein
3,Brown Rice,216,5,45,Carbs
4,Paneer,265,18,6,Protein
